In [5]:
import os
from openai import OpenAI
from google.colab import userdata

In [6]:
api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [1]:
from enum import Enum

class IntentCategory(str, Enum):
    FACTUAL_FAQ = "FACTUAL_FAQ"
    PROCEDURAL_GUIDANCE = "PROCEDURAL_GUIDANCE"
    COMMUNITY_EXPERIENCE = "COMMUNITY_EXPERIENCE"
    EVIDENCE_BASED = "EVIDENCE_BASED"
    COMPLEX_MULTI_HOP = "COMPLEX_MULTI_HOP"

In [2]:
from pydantic import BaseModel, Field

class IntentOutput(BaseModel):
    intent: IntentCategory = Field(
        description="The single most appropriate intent category for the user query"
    )

In [3]:
INTENT_CLASSIFIER_SYSTEM_PROMPT = """
You are an intent classification engine for a COVID-19 question-answering system.

Your task:
Given a user query, classify it into EXACTLY ONE intent category.

You must always return one category, even if the query does not clearly match any rules.
Use semantic understanding, not just keywords.

--------------------
INTENT CATEGORIES
--------------------

FACTUAL_FAQ:
- Short, direct factual questions
- Definitions, symptoms, transmission, vaccines, timelines
- Example: "What are the symptoms of COVID-19?"

PROCEDURAL_GUIDANCE:
- Questions asking for steps, actions, prevention, or advice
- Often includes "how", "what should I do", "can I", but do NOT rely only on keywords
- Example: "How can I protect myself from COVID-19?"

COMMUNITY_EXPERIENCE:
- Informal, opinionated, rumor-based, or social media influenced queries
- Mentions of people, beliefs, claims, or uncertainty
- Example: "People say masks don't work, is that true?"

EVIDENCE_BASED:
- Research-oriented, scientific, or evidence-seeking questions
- Mentions studies, data, research, clinical findings, or comparisons
- Example: "What does research say about COVID survival on surfaces?"

COMPLEX_MULTI_HOP:
- Long, multi-part, ambiguous, or mixed-intent questions
- Requires synthesising multiple facts or evidence
- Use this ONLY when no single intent clearly dominates
- Example: "How does COVID affect elderly people and what precautions should families take?"

--------------------
DECISION RULES (GUIDANCE, NOT HARD RULES)
--------------------

1. Use semantic meaning first, keywords second.
2. If multiple intents seem present:
   - Choose the MOST dominant intent.
3. If the query is long, compound, or unclear:
   - Prefer COMPLEX_MULTI_HOP.
4. If safety or scientific grounding is implied:
   - Prefer EVIDENCE_BASED over FACTUAL_FAQ.
5. Never return multiple categories.
6. Never invent new categories.
7. Never explain your reasoning.

Return ONLY the intent category in structured form.
"""


In [4]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.1 MB/s eta 0:00:00


In [7]:
from langchain_openai import ChatOpenAI

intent_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

intent_classifier = intent_llm.with_structured_output(IntentOutput)

In [8]:
def classify_intent_llm(query: str) -> IntentCategory:
    result = intent_classifier.invoke(
        [
            {"role": "system", "content": INTENT_CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ]
    )
    return result.intent

In [9]:
queries = [
    "What are the common symptoms of COVID-19?",
    "How should I isolate if I test positive?",
    "I read online that vaccines cause infertility, is it true?",
    "According to studies, how long does COVID survive on surfaces?",
    "Can COVID affect children differently and what precautions should parents take?"
]

for q in queries:
    print(q, "→", classify_intent_llm(q))


What are the common symptoms of COVID-19? → IntentCategory.FACTUAL_FAQ
How should I isolate if I test positive? → IntentCategory.PROCEDURAL_GUIDANCE
I read online that vaccines cause infertility, is it true? → IntentCategory.COMMUNITY_EXPERIENCE
According to studies, how long does COVID survive on surfaces? → IntentCategory.EVIDENCE_BASED
Can COVID affect children differently and what precautions should parents take? → IntentCategory.COMPLEX_MULTI_HOP
